# RAG System Demo
## Financial Document Question Answering

**System Mode:** CPU-first (GPU optional)

This notebook demonstrates the complete RAG pipeline for answering questions from 10-K filings.

**Requirements:**
- Python 3.8+
- 8-10GB RAM
- Works on CPU (GPU optional for 5-10x speedup)

### 1. Setup and Installation

**Note:** This installs CPU-compatible versions by default.

In [ ]:
import sys
sys.path.append('..')

import torch
from pathlib import Path
from loguru import logger

# Check system capabilities
print("="*60)
print("SYSTEM INFORMATION")
print("="*60)
print(f"GPU Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print("\n✅ GPU detected - you can use GPU mode for faster inference")
    print("   See CPU_SETUP.md for GPU configuration")
else:
    print("Device: CPU")
    print("\n✅ Running in CPU mode (default)")
    print("   Expected query time: 15-30 seconds")
    print("   RAM needed: 8-10GB")

print("="*60)

### 2. Initialize RAG Pipeline

**Default Configuration:**
- Uses CPU (safe for all systems)
- Model: Phi-3-mini (3.8B params)
- FAISS: CPU version
- Memory: 8-10GB RAM

In [ ]:
from src.config import RAGConfig
from src.main import initialize_pipeline, answer_question

print("Initializing RAG Pipeline with CPU-safe defaults...")
print("\nThis will:")
print("  1. Load PDF documents from data/pdfs/")
print("  2. Extract and chunk text")
print("  3. Generate embeddings (CPU)")
print("  4. Build FAISS index (CPU)")
print("  5. Load Phi-3-mini model")
print("\n⏱️  Expected time: 20-30 minutes on CPU\n")

initialize_pipeline()

print("\n✅ Pipeline initialized successfully!")

In [ ]:
# ALTERNATIVE: Explicitly use CPU-optimized config
# Uncomment to use this instead:

# from src.config_cpu import get_cpu_optimized_config
# config = get_cpu_optimized_config()
# print(f"Using config:")
# print(f"  Device: {config.llm.device}")
# print(f"  LLM: {config.llm.model_name}")
# print(f"  Embedding: {config.embedding.model_name}")
# print(f"  Batch size: {config.embedding.batch_size}")
# initialize_pipeline(config=config)

In [ ]:
# ALTERNATIVE: GPU config (only if you have GPU)
# Requires: pip uninstall faiss-cpu && pip install faiss-gpu
# Uncomment to use GPU:

# if torch.cuda.is_available():
#     from src.config import RAGConfig
#     config = RAGConfig()
#     config.use_gpu = True
#     config.embedding.device = "cuda"
#     config.llm.device = "cuda"
#     config.llm.model_name = "mistralai/Mistral-7B-Instruct-v0.2"
#     config.llm.load_in_4bit = True
#     config.vector_store.use_gpu = True
#     initialize_pipeline(config=config)
# else:
#     print("⚠️ No GPU available. Using CPU config.")
#     initialize_pipeline()

### 3. Ask Questions

Now you can ask questions about Apple and Tesla 10-K filings.

**Expected response time:**
- CPU: 15-30 seconds per question
- GPU: 2-3 seconds per question

In [ ]:
# Example 1: Apple revenue question
import time

question = "What was Apple's total revenue for fiscal year 2024?"
print(f"Question: {question}")
print("\n⏱️  Processing (this may take 15-30 seconds on CPU)...\n")

start_time = time.time()
result = answer_question(question)
elapsed = time.time() - start_time

print("="*60)
print(f"Answer: {result['answer']}")
print(f"\nSources: {result['sources']}")
print(f"\nTime taken: {elapsed:.2f} seconds")
print("="*60)

In [ ]:
# Example 2: Tesla question
question = "What types of vehicles does Tesla currently produce and deliver?"
print(f"Question: {question}\n")

start_time = time.time()
result = answer_question(question)
elapsed = time.time() - start_time

print("="*60)
print(f"Answer: {result['answer']}")
print(f"\nSources: {result['sources']}")
print(f"\nTime taken: {elapsed:.2f} seconds")
print("="*60)

In [ ]:
# Example 3: Out-of-scope question (should refuse)
question = "What is Tesla's stock price forecast for 2025?"
print(f"Question: {question}\n")

result = answer_question(question)

print("="*60)
print(f"Answer: {result['answer']}")
print(f"\nSources: {result['sources']}")
print("\n✅ Should refuse with proper message (not in documents)")
print("="*60)

### 4. Batch Processing

Process multiple questions efficiently.

In [ ]:
from src.main import answer_questions_batch

test_questions = [
    {"question_id": 1, "question": "What was Apple's total revenue for fiscal year 2024?"},
    {"question_id": 6, "question": "What was Tesla's total revenue for 2023?"},
    {"question_id": 9, "question": "What types of vehicles does Tesla produce?"},
    {"question_id": 11, "question": "What is Tesla's stock price forecast for 2025?"}
]

print("Processing batch of questions...")
print(f"Total questions: {len(test_questions)}")
print(f"⏱️  Estimated time: {len(test_questions) * 20} - {len(test_questions) * 30} seconds on CPU\n")

start_time = time.time()
results = answer_questions_batch(test_questions)
elapsed = time.time() - start_time

print("\n" + "="*60)
print("RESULTS")
print("="*60)
for r in results:
    print(f"\nQ{r['question_id']}: {r['answer'][:100]}...")
    print(f"Sources: {r['sources']}")

print(f"\n⏱️  Total time: {elapsed:.2f} seconds")
print(f"   Average: {elapsed/len(test_questions):.2f} seconds per question")
print("="*60)

### 5. Evaluation

Evaluate the quality of answers including hallucination detection.

In [ ]:
from src.evaluation import RAGEvaluator

# Evaluate the batch results
evaluator = RAGEvaluator()
metrics = evaluator.evaluate(results)

# Print detailed report
evaluator.print_report(metrics)

In [ ]:
# View specific metrics
import json

print("\nAGGREGATE METRICS:")
print(json.dumps(metrics['aggregate'], indent=2))

### 6. Custom Questions

In [ ]:
# Interactive question answering
def ask_interactive():
    """Interactive Q&A loop."""
    print("\n" + "="*60)
    print("INTERACTIVE MODE")
    print("="*60)
    print("Ask questions about Apple or Tesla 10-K filings.")
    print("Type 'quit' to exit.\n")
    
    while True:
        question = input("\nYour question: ").strip()
        
        if question.lower() in ['quit', 'exit', 'q']:
            print("\nGoodbye!")
            break
        
        if not question:
            continue
        
        print("\n⏱️  Processing...\n")
        start_time = time.time()
        result = answer_question(question)
        elapsed = time.time() - start_time
        
        print("="*60)
        print(f"Answer: {result['answer']}")
        print(f"\nSources: {result['sources']}")
        print(f"\nTime: {elapsed:.2f}s")
        print("="*60)

# Uncomment to run interactive mode
# ask_interactive()

In [ ]:
# Or ask a single custom question
custom_question = "How many shares of common stock did Apple have outstanding?"

result = answer_question(custom_question)

print(f"Question: {custom_question}")
print(f"\nAnswer: {result['answer']}")
print(f"\nSources: {result['sources']}")

### 7. Pipeline Statistics

View information about the RAG pipeline.

In [ ]:
from src.main import _pipeline

if _pipeline:
    stats = _pipeline.get_pipeline_stats()
    
    print("="*60)
    print("PIPELINE STATISTICS")
    print("="*60)
    
    print(f"\nIndexed: {stats['is_indexed']}")
    print(f"\nConfiguration:")
    for key, value in stats['config'].items():
        print(f"  {key}: {value}")
    
    if 'vector_store' in stats:
        print(f"\nVector Store:")
        for key, value in stats['vector_store'].items():
            print(f"  {key}: {value}")
    
    print("="*60)
else:
    print("Pipeline not initialized yet.")